In [3]:
import pandas as pd
import numpy as np

# models
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# tree-based models
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [4]:
df = pd.read_csv("/Users/julia/df_reg_cleaned.csv")

### Injury Count

In [5]:
# Y value is injury count 
y = df['injury_count']
print(len(y))
print(y.value_counts())

226969
injury_count
0     123398
1      71473
2      21434
3       6686
4       2344
5        918
6        393
7        171
8         74
9         31
10        17
12        10
11         7
15         3
13         3
14         2
21         1
18         1
22         1
61         1
17         1
Name: count, dtype: int64


In [6]:
# X value is everything except injury count 
x = df.drop(columns=['injury_count'])
print(len(x))
print(x.value_counts())

226969
crash_speed_limit  road_constr_zone_fl  latitude   longitude   onsys_fl  private_dr_fl  crash_speed_limit_missing  collision_type_grouped  num_units  has_pedestrian  has_motorcycle  has_bicycle  has_micromobility  has_large_vehicle  has_other  hour  day_of_week  time_bucket 
60.0               False                30.437280  -97.735875  1         0              0                          angle                   2          0               0               0            0                  1                  0          12    4            midday          3
45.0               False                30.347687  -97.712448  1         0              1                          angle                   2          0               0               0            0                  1                  0          12    3            midday          3
30.0               False                30.375338  -97.674404  0         0              0                          angle                   2          0    

### Log Transformation

In [7]:
y_log = np.log1p(y)

print(y_log)

0         0.000000
1         0.000000
2         0.000000
3         0.000000
4         0.693147
            ...   
226964    1.098612
226965    0.000000
226966    1.098612
226967    0.000000
226968    0.693147
Name: injury_count, Length: 226969, dtype: float64


In [8]:
comparison = pd.DataFrame({
    'Original': y,
    'Log_Transformed': y_log
})

print(comparison)

        Original  Log_Transformed
0              0         0.000000
1              0         0.000000
2              0         0.000000
3              0         0.000000
4              1         0.693147
...          ...              ...
226964         2         1.098612
226965         0         0.000000
226966         2         1.098612
226967         0         0.000000
226968         1         0.693147

[226969 rows x 2 columns]


### Comparing with original in Summary Stats 

In [9]:
print("Original:")
print(y.describe())

print("\nLog Transformed:")
print(y_log.describe())

Original:
count    226969.000000
mean          0.675903
std           0.972693
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          61.000000
Name: injury_count, dtype: float64

Log Transformed:
count    226969.000000
mean          0.393230
std           0.469586
min           0.000000
25%           0.000000
50%           0.000000
75%           0.693147
max           4.127134
Name: injury_count, dtype: float64


#### The result shows that the maximum value decreases a lot and distribution becomes more balanced. Therefore, the injury count was highly right skewed with many low values and a few extreme high values. After applying a log transformation, the distribution of the data becomes more balanced since extreme values are compressed. Overall, log transformation helps to stabilize the data compare to the orginal data.

## Linear Regression 

In [19]:
df = df.dropna()
df = df.select_dtypes(include='number')

x = df.drop(columns=['injury_count'])
y = df['injury_count']

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=15)

def adjusted_r2(r2, n, p):
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))

lr = LinearRegression()
lr.fit(x_train, y_train)


y_pred = lr.predict(x_test)

In [21]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
adj_r2 = adjusted_r2(r2, x_test.shape[0], x_test.shape[1])

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Adjusted R2:", adj_r2)

MAE: 0.7157257262451592
RMSE: 1.0029510569404803
R2: 0.01292478239393291
Adjusted R2: 0.012598498109387735


### The result shows me that the regression model is not useful for predicting injuries. The R² value of about 0.013 means the model explains only around 1% of the variation in injury counts, so nearly all of the changes in injury count are not captured by your model. The adjusted R² being almost the same confirms that adding more variables did not improve the model. The RMSE of about 1.0 and MAE of about 0.7 show that your predictions are off by roughly 1 injury on average, which can be a large error if the counts themselves are small. Overall, this means the model is only slightly better than simply guessing the average injury count for everyone. The main issue is that the model is not finding strong relationships between your features and the target variable, which could be because the predictors are weak, the relationship is not linear, or injury count (as a count variable) is not well suited for linear regression. Although the model runs properly, it is not useful for accurate prediction and interpretation. 